# Feature-based RBF residual GRAPE

This notebook tests the next residual idea:

```text
u -> nominal simulated trajectory -> physics-informed features -> RBF correction
```

The older RBF saw the raw 80 B-spline coefficients. That is hard because two pulses can be far in coefficient space but physically similar. Here the RBF sees lower-dimensional features such as integrated qubit excitation, photon-number timing, drive power, and pulse smoothness.

The optimizer still uses JAX autodifferentiation through the physics simulator and the learned correction.

In [ ]:
from dataclasses import replace

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from hybrid_residual_grape import (
    FEATURE_NAMES,
    FockPhysicsModel,
    HybridGrapeConfig,
    PhysicsParams,
    RBFResidualConfig,
    SimulationConfig,
    append_dataset,
    beta_normal_lower_bound,
    empty_feature_rbf_model,
    feature_hybrid_probability_batch,
    fit_feature_rbf_residual,
    optimize_feature_hybrid_grape,
    sample_binomial_measurements,
    trajectory_feature_matrix,
)
from hybrid_residual_grape.config import (
    khz_to_rad_per_us,
    qubit_t1_us,
    qubit_t2_us,
    storage_t1_us,
    storage_t2_us,
)
from hybrid_residual_grape.residual import measured_probability

jax.config.update("jax_enable_x64", True)


## Setup

`physics_model` is the model used by GRAPE and by the feature extractor. `true_model` is only the hidden simulator used here to mimic the experiment. On hardware, `measure_on_experiment` should be replaced by the OPX measurement call.

The hidden model includes small detunings, drive-scale errors, cavity self-Kerr, and shortened T1/T2. The nominal model omits these effects.

This notebook uses `mu_qub = mu_cav = 40.0` while keeping the coefficient clip at `[-2, 2]`. This tests whether the previous runs were partly limited by the pulse coefficients saturating at the amplitude bound.

In [ ]:
seed = 2026
key = jax.random.key(seed)

q = SimulationConfig(
    n_cav=25,
    target_n=2,
    initial_cavity_n=0,
    initial_qubit_state=0,
    t_drive=1.408,
    ndt_drive=80,
    num_coeffs=20,
    spline_degree=2,
    spline_skip_left=2,
    spline_skip_right=2,
    param_clip=2.0,
)

# Larger calibrated drive conversion: this keeps the hardware/playback coefficient
# bound at [-2, 2], but makes the same coefficients generate stronger dynamics.
# If the real amp-to-Rabi calibration is closer to 20, set this back to 20.
drive_coupling = 40.0
config_params = replace(PhysicsParams(), mu_qub=drive_coupling, mu_cav=drive_coupling)

nominal_params = replace(
    config_params,
    cavity_self_kerr=0.0,
    qubit_t1_us=None,
    qubit_t2_us=None,
    cavity_t1_us=None,
    cavity_t2_us=None,
)
physics_model = FockPhysicsModel(q, nominal_params)

lifetime_scale = 0.75
true_params = replace(
    config_params,
    chi=config_params.chi + khz_to_rad_per_us(3.0),
    cavity_self_kerr=config_params.cavity_self_kerr + khz_to_rad_per_us(0.12),
    cavity_detuning=khz_to_rad_per_us(5.0),
    qubit_detuning=khz_to_rad_per_us(-5.0),
    mu_qub=config_params.mu_qub * 1.010,
    mu_cav=config_params.mu_cav * 0.988,
    cavity_phase=0.025,
    qubit_t1_us=lifetime_scale * qubit_t1_us(),
    qubit_t2_us=lifetime_scale * qubit_t2_us(),
    cavity_t1_us=lifetime_scale * storage_t1_us(),
    cavity_t2_us=lifetime_scale * storage_t2_us(),
)
true_model = FockPhysicsModel(q, true_params)

print("parameter size:", physics_model.parameter_size)
print("mu_qub/mu_cav:", config_params.mu_qub, config_params.mu_cav)
print("feature size:", len(FEATURE_NAMES))
print("features:")
for name in FEATURE_NAMES:
    print(" -", name)
print("hidden collapse operators:", len(true_model.collapse_ops))

## Measurement hook

This function currently samples binomial counts from the hidden simulator. The optimizer only sees `successes / shots`, not the hidden true probability.

In [ ]:
shot_time_us = 400.0


def measure_on_experiment(controls, key, shots):
    return sample_binomial_measurements(true_model, controls, key, shots=shots)

## Hyperparameters

The feature RBF length scale is in standardized feature units. A value around `1` means that points are considered similar if their trajectory features are within about one empirical standard deviation.

The residual is applied in **logit probability**. Near high fidelity, a modest probability error can be a large logit error. For example, correcting a nominal prediction near `0.999` down to a measured value near `0.94` requires a logit correction of several units. Therefore the residual clip must be large enough not to saturate during normal operation, and the GRAPE objective should not strongly penalize the learned residual magnitude itself. The safety mechanism should mainly be the support penalty, i.e. distrust unsupported extrapolation.

In [ ]:

feature_rbf_config = RBFResidualConfig(
    max_centers=256,
    length_scale=0.75,
    ridge=1e-2,
    residual_clip=6.0,
    measurement_floor=1e-3,
)

baseline_config = HybridGrapeConfig(
    maxiter=320,
    memory_size=20,
    noise_samples=1,
    control_noise_std=0.0,
    residual_support_penalty=0.0,
    residual_size_penalty=0.0,
    amplitude_l2=3e-5,
    smoothness_l2=1e-4,
)

feature_grape_config = HybridGrapeConfig(
    maxiter=180,
    memory_size=18,
    noise_samples=5,
    control_noise_std=0.015,
    residual_support_penalty=0.15,
    residual_size_penalty=0.0,
    amplitude_l2=3e-5,
    smoothness_l2=1e-4,
)

num_rounds = 48

# Staged experimental search around the current best measured pulse.
screening_shots = 300
validation_shots = 2500
validation_top_k = 3
selection_z = 1.0
accept_z = 1.0
acceptance_margin = 0.0

# Candidate pool: anchor + surrogate-GRAPE + line search + SPSA pairs + Gaussian perturbations.
line_candidate_count = 4
spsa_pairs = 4
random_best_count = 4
random_surrogate_count = 3
spsa_step = 0.04
local_noise_std = 0.035
surrogate_noise_std = 0.020
candidate_pool_size = 2 + line_candidate_count + 2 * spsa_pairs + random_best_count + random_surrogate_count
validated_per_round = 1 + validation_top_k

screening_measurements = candidate_pool_size * screening_shots
validation_measurements = validated_per_round * validation_shots
print("candidate pool size:", candidate_pool_size)
print("validated candidates per round:", validated_per_round)
print("measurements per round:", screening_measurements + validation_measurements)
print("measurement time per round [s]:", (screening_measurements + validation_measurements) * shot_time_us / 1e6)


## Baseline: pure GRAPE on the nominal model

This is the comparison point: optimize the incomplete physics model, then evaluate the resulting pulse on the hidden true model.

In [ ]:
key, init_key, grape_key = jax.random.split(key, 3)
initial_controls = 0.12 * jax.random.normal(init_key, (physics_model.parameter_size,))
initial_controls = jnp.clip(initial_controls, -0.4, 0.4)

empty_feature_residual = empty_feature_rbf_model(feature_rbf_config)
pure_controls, pure_history, pure_summary, key = optimize_feature_hybrid_grape(
    physics_model,
    empty_feature_residual,
    initial_controls,
    grape_key,
    baseline_config,
)

pure_physics = float(physics_model.photon_probability(pure_controls))
pure_true = float(true_model.photon_probability(pure_controls))
print("pure nominal-GRAPE predicted P_n:", pure_physics)
print("pure nominal-GRAPE hidden true P_n:", pure_true)


## Closed-loop feature-RBF GRAPE with measured acceptance

The previous version let the learned hybrid model propose one pulse, measured nearby perturbations, then started the next round from that proposed pulse. That can drift when the residual model is overconfident. The loop below is intentionally more conservative:

1. Keep an **anchor pulse**: the best pulse that has been validated by binary-shot measurements.
2. Run GRAPE on the current hybrid model, starting from the anchor, to get a **surrogate proposal**.
3. Build a small candidate pool around the anchor and proposal:
   - the anchor itself,
   - the surrogate-GRAPE proposal,
   - points on the line between them,
   - SPSA-style paired perturbations around the anchor,
   - Gaussian local perturbations around the anchor and proposal.
4. Give every candidate a cheap screening measurement.
5. Give the anchor and the best screened challengers extra validation shots.
6. Move the anchor only if a challenger has a better lower-confidence score.
7. Add all measurements to the dataset, refit the feature-RBF correction, and repeat.

This is closer to the real experiment logic: the learned model proposes where to look, but measured data decides whether we actually trust the move.

### Reading the training output

Each printed line is one closed-loop round, not an average over independent notebook runs.

- `anchor=a/b`: measured mean `a` and lower-confidence score `b` for the currently accepted pulse. This value accumulates repeated validation shots across rounds.
- `challenger=a/b`: measured mean and lower-confidence score for the best newly tested challenger in that round.
- `accept=1`: the challenger replaced the anchor. `accept=0`: we stayed with the old anchor.
- `surrogate`: hybrid-model prediction for the GRAPE proposal **before** measuring it in the current round. This is the honest out-of-sample model prediction.
- `support`: feature-RBF support for that surrogate proposal before measurement. Near 1 means the proposal lies near previously measured feature points; near 0 means extrapolation.
- `best_true`: hidden simulated true `P_n`, only available in this notebook. In the lab this number does not exist.
- `data`: number of measured pulse records in the dataset. Each record can have many binary shots.

The plots keep both pre-measurement and post-refit quantities. Post-refit predictions are useful diagnostics, but they are not evidence of successful generalization by themselves, because the just-measured pulse has already been added to the training set.


In [ ]:

def clip_controls(controls):
    return jnp.clip(controls, -q.param_clip, q.param_clip)


def make_candidate_pool(anchor_controls, surrogate_controls, key):
    key, spsa_key, best_noise_key, surrogate_noise_key = jax.random.split(key, 4)
    anchor_controls = jnp.asarray(anchor_controls)
    surrogate_controls = jnp.asarray(surrogate_controls)

    anchor = anchor_controls[None, :]
    surrogate = surrogate_controls[None, :]

    line_fracs = jnp.linspace(0.2, 0.8, line_candidate_count)[:, None]
    line = anchor_controls[None, :] + line_fracs * (surrogate_controls - anchor_controls)[None, :]

    spsa_dirs = jax.random.rademacher(spsa_key, (spsa_pairs, anchor_controls.shape[0]))
    spsa = jnp.concatenate(
        [
            anchor_controls[None, :] + spsa_step * spsa_dirs,
            anchor_controls[None, :] - spsa_step * spsa_dirs,
        ],
        axis=0,
    )

    best_random = anchor_controls[None, :] + local_noise_std * jax.random.normal(
        best_noise_key,
        (random_best_count, anchor_controls.shape[0]),
    )
    surrogate_random = surrogate_controls[None, :] + surrogate_noise_std * jax.random.normal(
        surrogate_noise_key,
        (random_surrogate_count, anchor_controls.shape[0]),
    )

    pool = jnp.concatenate([anchor, surrogate, line, spsa, best_random, surrogate_random], axis=0)
    return clip_controls(pool), key


def select_validation_indices(screen_scores):
    order = [int(i) for i in jnp.asarray(jnp.argsort(screen_scores)[::-1]).tolist()]
    challengers = [idx for idx in order if idx != 0][:validation_top_k]
    return jnp.array([0] + challengers, dtype=jnp.int32)


def append_measurements(controls, successes, shots):
    global dataset_controls, dataset_successes, dataset_shots, dataset_physics
    physics_probs = physics_model.population_probability(controls)
    dataset_controls, dataset_successes, dataset_shots, dataset_physics = append_dataset(
        dataset_controls,
        dataset_successes,
        dataset_shots,
        dataset_physics,
        controls,
        successes,
        shots,
        physics_probs,
    )
    return int(jnp.sum(shots))


dataset_controls = None
dataset_successes = None
dataset_shots = None
dataset_physics = None

records = []
feature_snapshots = []
total_measurements = 0

# Initial anchor validation. In the real experiment this is the first pulse we trust.
key, init_measure_key = jax.random.split(key)
init_successes, init_shots, init_true_prob, key = measure_on_experiment(
    pure_controls[None, :],
    init_measure_key,
    shots=validation_shots,
)
total_measurements += append_measurements(pure_controls[None, :], init_successes, init_shots)

best_controls = pure_controls
best_successes = init_successes[0]
best_shots = init_shots[0]
best_measured = float(best_successes / best_shots)
best_score = float(beta_normal_lower_bound(best_successes, best_shots, z=accept_z))
anchor_true_diagnostic = float(init_true_prob[0])
best_seen_true_diagnostic = anchor_true_diagnostic

residual_model, dataset_features, dataset_physics_from_features = fit_feature_rbf_residual(
    physics_model,
    dataset_controls,
    dataset_successes,
    dataset_shots,
    feature_rbf_config,
)

print(
    f"initial anchor measured={best_measured:.4f} lcb={best_score:.4f} "
    f"true_diagnostic={anchor_true_diagnostic:.4f} shots={int(best_shots)}"
)


In [ ]:

for round_idx in tqdm(range(num_rounds), desc="feature-RBF trust-region rounds"):
    key, grape_key, pool_key, screen_key, validation_key = jax.random.split(key, 5)

    surrogate_controls, grape_history, summary, key = optimize_feature_hybrid_grape(
        physics_model,
        residual_model,
        best_controls,
        grape_key,
        feature_grape_config,
    )
    surrogate_pred, surrogate_physics, surrogate_support, surrogate_residual, surrogate_features = feature_hybrid_probability_batch(
        physics_model,
        residual_model,
        surrogate_controls[None, :],
    )

    pool_controls, key = make_candidate_pool(best_controls, surrogate_controls, pool_key)
    screen_successes, screen_shots, screen_true_probs, key = measure_on_experiment(
        pool_controls,
        screen_key,
        shots=screening_shots,
    )
    total_measurements += append_measurements(pool_controls, screen_successes, screen_shots)

    screen_scores = beta_normal_lower_bound(screen_successes, screen_shots, z=selection_z)
    validation_indices = select_validation_indices(screen_scores)
    validation_controls = pool_controls[validation_indices]
    validation_successes, validation_shots_arr, validation_true_probs, key = measure_on_experiment(
        validation_controls,
        validation_key,
        shots=validation_shots,
    )
    total_measurements += append_measurements(
        validation_controls,
        validation_successes,
        validation_shots_arr,
    )

    staged_successes = screen_successes[validation_indices] + validation_successes
    staged_shots = screen_shots[validation_indices] + validation_shots_arr
    staged_measured = staged_successes / staged_shots
    staged_scores = beta_normal_lower_bound(staged_successes, staged_shots, z=accept_z)

    # The current anchor is always validation index 0. Accumulate its repeated shots.
    best_successes = best_successes + staged_successes[0]
    best_shots = best_shots + staged_shots[0]
    best_measured = float(best_successes / best_shots)
    best_score = float(beta_normal_lower_bound(best_successes, best_shots, z=accept_z))

    if validation_indices.shape[0] > 1:
        challenger_local = 1 + int(jnp.argmax(staged_scores[1:]))
    else:
        challenger_local = 0
    challenger_pool_idx = int(validation_indices[challenger_local])
    challenger_measured = float(staged_measured[challenger_local])
    challenger_score = float(staged_scores[challenger_local])
    challenger_true_diagnostic = float(screen_true_probs[challenger_pool_idx])

    accepted = bool(challenger_local != 0 and challenger_score > best_score + acceptance_margin)
    if accepted:
        best_controls = pool_controls[challenger_pool_idx]
        best_successes = staged_successes[challenger_local]
        best_shots = staged_shots[challenger_local]
        best_measured = float(best_successes / best_shots)
        best_score = float(beta_normal_lower_bound(best_successes, best_shots, z=accept_z))
        anchor_true_diagnostic = challenger_true_diagnostic

    round_best_true = float(jnp.max(screen_true_probs))
    best_seen_true_diagnostic = max(best_seen_true_diagnostic, round_best_true)

    residual_model, dataset_features, dataset_physics_from_features = fit_feature_rbf_residual(
        physics_model,
        dataset_controls,
        dataset_successes,
        dataset_shots,
        feature_rbf_config,
    )
    hybrid_train, physics_train, support_train, residual_train, _ = feature_hybrid_probability_batch(
        physics_model,
        residual_model,
        dataset_controls,
    )
    measured_train = measured_probability(
        dataset_successes,
        dataset_shots,
        feature_rbf_config.measurement_floor,
    )
    post_surrogate_pred, _, post_surrogate_support, post_surrogate_residual, _ = feature_hybrid_probability_batch(
        physics_model,
        residual_model,
        surrogate_controls[None, :],
    )
    anchor_pred, anchor_physics, anchor_support, anchor_residual, anchor_features = feature_hybrid_probability_batch(
        physics_model,
        residual_model,
        best_controls[None, :],
    )

    records.append(
        {
            "round": round_idx,
            "total_measurements": total_measurements,
            "accepted": float(accepted),
            "pool_size": int(pool_controls.shape[0]),
            "validated_count": int(validation_indices.shape[0]),
            "anchor_measured": best_measured,
            "anchor_score": best_score,
            "anchor_true_diagnostic": anchor_true_diagnostic,
            "challenger_measured": challenger_measured,
            "challenger_score": challenger_score,
            "challenger_true_diagnostic": challenger_true_diagnostic,
            "round_best_true_diagnostic": round_best_true,
            "best_seen_true_diagnostic": best_seen_true_diagnostic,
            "surrogate_pred_hybrid": float(surrogate_pred[0]),
            "surrogate_pred_physics": float(surrogate_physics[0]),
            "surrogate_support": float(surrogate_support[0]),
            "surrogate_residual_logit": float(surrogate_residual[0]),
            "post_surrogate_pred_hybrid": float(post_surrogate_pred[0]),
            "post_surrogate_support": float(post_surrogate_support[0]),
            "anchor_pred_hybrid": float(anchor_pred[0]),
            "anchor_pred_physics": float(anchor_physics[0]),
            "anchor_support": float(anchor_support[0]),
            "anchor_residual_logit": float(anchor_residual[0]),
            "train_mae_physics": float(jnp.mean(jnp.abs(physics_train - measured_train))),
            "train_mae_hybrid": float(jnp.mean(jnp.abs(hybrid_train - measured_train))),
            "train_clip_fraction": float(
                jnp.mean(
                    (jnp.abs(residual_train) > 0.98 * feature_rbf_config.residual_clip).astype(jnp.float32)
                )
            ),
            "data_size": int(dataset_controls.shape[0]),
        }
    )
    feature_snapshots.append(anchor_features[0])

    tqdm.write(
        f"round {round_idx:02d} "
        f"anchor={best_measured:.4f}/{best_score:.4f} "
        f"challenger={challenger_measured:.4f}/{challenger_score:.4f} "
        f"accept={int(accepted)} "
        f"surrogate={float(surrogate_pred[0]):.4f} "
        f"support={float(surrogate_support[0]):.3f} "
        f"best_true={best_seen_true_diagnostic:.4f} "
        f"data={dataset_controls.shape[0]}"
    )


## Diagnostics

In [ ]:

xs = jnp.array([r["total_measurements"] for r in records])

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

axes[0, 0].plot(xs, [r["anchor_true_diagnostic"] for r in records], "o-", label="accepted anchor true diagnostic")
axes[0, 0].plot(xs, [r["best_seen_true_diagnostic"] for r in records], "o-", label="best seen true diagnostic")
axes[0, 0].axhline(pure_true, color="black", linestyle="--", label="pure GRAPE true")
axes[0, 0].set_xlabel("total binary measurements")
axes[0, 0].set_ylabel("P_n")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(xs, [r["surrogate_pred_hybrid"] for r in records], "o-", label="surrogate prediction before measurement")
axes[0, 1].plot(xs, [r["post_surrogate_pred_hybrid"] for r in records], "o-", label="surrogate prediction after refit")
axes[0, 1].plot(xs, [r["challenger_true_diagnostic"] for r in records], "o-", label="best challenger true diagnostic")
axes[0, 1].set_xlabel("total binary measurements")
axes[0, 1].set_ylabel("P_n")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(xs, [r["anchor_measured"] for r in records], "o-", label="anchor measured mean")
axes[1, 0].plot(xs, [r["anchor_score"] for r in records], "o-", label="anchor lower-confidence score")
axes[1, 0].plot(xs, [r["challenger_measured"] for r in records], "o-", label="challenger measured mean")
axes[1, 0].plot(xs, [r["challenger_score"] for r in records], "o-", label="challenger lower-confidence score")
axes[1, 0].set_xlabel("total binary measurements")
axes[1, 0].set_ylabel("measured P_n")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(xs, [r["surrogate_support"] for r in records], "o-", label="surrogate support before measurement")
axes[1, 1].plot(xs, [r["anchor_support"] for r in records], "o-", label="accepted anchor support after refit")
axes[1, 1].plot(xs, [r["accepted"] for r in records], "o", label="accepted this round")
axes[1, 1].plot(xs, [r["train_clip_fraction"] for r in records], "o-", label="fraction clipped")
axes[1, 1].set_xlabel("total binary measurements")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.show()

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(xs, [r["train_mae_physics"] for r in records], "o-", label="physics -> measured")
ax.plot(xs, [r["train_mae_hybrid"] for r in records], "o-", label="hybrid -> measured")
ax.set_xlabel("total binary measurements")
ax.set_ylabel("training MAE")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


In [ ]:
feature_array = jnp.stack(feature_snapshots)
fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
im = ax.imshow(feature_array.T, aspect="auto", interpolation="nearest")
ax.set_yticks(range(len(FEATURE_NAMES)))
ax.set_yticklabels(FEATURE_NAMES)
ax.set_xlabel("closed-loop round")
ax.set_title("features of the accepted anchor pulse each round")
fig.colorbar(im, ax=ax)
plt.show()
